In [1]:
#文件系统
import os

#加载.env
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

#导入文件读取器
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

#导入文档分割器
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#导入质谱Embedding
from zhipuai_embedding import ZhipuAIEmbeddings

#导入chroma
from langchain_community.vectorstores import Chroma

import re

In [2]:

# 获取folder_path下所有文件路径，储存在file_paths里
file_paths = []
folder_path = "./llm-universe/data_base/knowledge_db"
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

['./llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.mp4', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.vtt', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.txt', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.srt', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.json', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.tsv', './llm-universe/data_base/knowledge_db/prompt_engineering/6. 文本转换 Transforming.md', './llm-universe/data_base/knowledge_db/prompt_engineering/3. 迭代优化 Iterative.md', './llm-universe/data_base/knowledge_db/prompt_engineering/7. 文本扩展 Expanding.md', './llm-universe/data_base/knowledge_db/prompt_engineering/8. 聊天机器人 Chatbot.md', './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md', './llm-universe/data_base/knowledge_db/prompt_engineering/2. 提示原则 Guidelines.md', './llm-universe/data_base/knowledge_db/prompt_engineering/9. 总结 Summa

In [3]:
#把pdf实例化后保存,md文档则直接读取
Loaders=[]
texts=[]
md_paths=[]
for file in file_paths:
    file_type=file.split(".")[-1]
    if file_type=="pdf":
        Loaders.append(PyMuPDFLoader(file))
    elif file_type=="md":
        # Loaders.append(UnstructuredMarkdownLoader(file))----这里实例化后导致md文档中标题的#符号消失
        with open(file,"r",encoding="utf-8") as f:
            texts.append(f.read())#此操作会优先将md文档保存在列表的前面部分
            md_paths.append(file)
            


In [4]:
#选取一份md文档进行清洗和分割
for loader in Loaders:
    texts.extend(loader.load())
text=texts[4]
print(text)

# 第五章 推断

在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。

让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的情感和主题。这些任务包括了标签提取、实体提取、以及理解文本的情感等等。在传统的机器学习流程中，你需要收集标签化的数据集、训练模型、确定如何在云端部署模型并进行推断。尽管这种方式可能会产生不错的效果，但完成这一全流程需要耗费大量的时间和精力。而且，每一个任务，比如情感分析、实体提取等等，都需要训练和部署单独的模型。

然而，就在你准备投入繁重工作的时候，你发现了大型语言模型（LLM）。LLM 的一个明显优点是，对于许多这样的任务，你只需要编写一个 Prompt，就可以开始生成结果，大大减轻了你的工作负担。这个发现像是找到了一把神奇的钥匙，让应用程序开发的速度加快了许多。最令你兴奋的是，你可以仅仅使用一个模型和一个 API 来执行许多不同的任务，无需再纠结如何训练和部署许多不同的模型。

让我们开始这一章的学习，一起探索如何利用 LLM 加快我们的工作进程，提高我们的工作效率。

## 一、情感推断

### 1.1 情感倾向分析

让我们以一则电商平台上的台灯评论为例，通过此例，我们将学习如何对评论进行情感二分类（正面/负面）。


```python
lamp_review = """
我需要一盏漂亮的卧室灯，这款灯具有额外的储物功能，价格也不算太高。\
我很快就收到了它。在运输过程中，我们的灯绳断了，但是公司很乐意寄送了一个新的。\
几天后就收到了。这款灯很容易组装。我发现少了一个零件，于是联系了他们的客服，他们很快就给我寄来了缺失的零件！\
在我看来，Lumina 是一家非常关心顾客和产品的优秀公司！
"""
```

接下来，我们将尝试编写一个 Prompt ，用以分类这条商品评论的情感。如果我们想让系统解析这条评论的情感倾向，只需编写“以下商品评论的情感倾向是什么？”这样的 Prompt ，再加上一些标准的分隔符和评论文本等。

然后，我们将这个程序运行一遍。结果表明，这条商品评论的情感倾向是正面的，这似乎非常准确。尽管这款台灯并非完美无缺，但是这位顾客对它似乎相当满意。这个公司看起来非常重视客户体验和产品质量，因此，认定评论的情感倾向为正面似乎是正确的判

In [5]:
#轻度数据清洗
text=re.sub(r"\n{3,}","\n\n",text)
text=re.sub(r"# 中文","",text)
print(text)

# 第五章 推断

在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。

让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的情感和主题。这些任务包括了标签提取、实体提取、以及理解文本的情感等等。在传统的机器学习流程中，你需要收集标签化的数据集、训练模型、确定如何在云端部署模型并进行推断。尽管这种方式可能会产生不错的效果，但完成这一全流程需要耗费大量的时间和精力。而且，每一个任务，比如情感分析、实体提取等等，都需要训练和部署单独的模型。

然而，就在你准备投入繁重工作的时候，你发现了大型语言模型（LLM）。LLM 的一个明显优点是，对于许多这样的任务，你只需要编写一个 Prompt，就可以开始生成结果，大大减轻了你的工作负担。这个发现像是找到了一把神奇的钥匙，让应用程序开发的速度加快了许多。最令你兴奋的是，你可以仅仅使用一个模型和一个 API 来执行许多不同的任务，无需再纠结如何训练和部署许多不同的模型。

让我们开始这一章的学习，一起探索如何利用 LLM 加快我们的工作进程，提高我们的工作效率。

## 一、情感推断

### 1.1 情感倾向分析

让我们以一则电商平台上的台灯评论为例，通过此例，我们将学习如何对评论进行情感二分类（正面/负面）。

```python
lamp_review = """
我需要一盏漂亮的卧室灯，这款灯具有额外的储物功能，价格也不算太高。\
我很快就收到了它。在运输过程中，我们的灯绳断了，但是公司很乐意寄送了一个新的。\
几天后就收到了。这款灯很容易组装。我发现少了一个零件，于是联系了他们的客服，他们很快就给我寄来了缺失的零件！\
在我看来，Lumina 是一家非常关心顾客和产品的优秀公司！
"""
```

接下来，我们将尝试编写一个 Prompt ，用以分类这条商品评论的情感。如果我们想让系统解析这条评论的情感倾向，只需编写“以下商品评论的情感倾向是什么？”这样的 Prompt ，再加上一些标准的分隔符和评论文本等。

然后，我们将这个程序运行一遍。结果表明，这条商品评论的情感倾向是正面的，这似乎非常准确。尽管这款台灯并非完美无缺，但是这位顾客对它似乎相当满意。这个公司看起来非常重视客户体验和产品质量，因此，认定评论的情感倾向为正面似乎是正确的判断

In [6]:
#按照章节标题进行切分
headers_to_split_on=[
    ("#","一级标题"),
    ("##","二级标题"),
    ("###","三级标题")
]
header_spliter=MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
header_docs=header_spliter.split_text(text)
print(len(header_docs))

10


In [7]:
#添加文件来源
for doc in header_docs:
    doc.metadata["source"]=md_paths[4]
    print(doc.metadata)

{'一级标题': '第五章 推断', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '一、情感推断', '三级标题': '1.1 情感倾向分析', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '一、情感推断', '三级标题': '1.2 识别情感类型', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '一、情感推断', '三级标题': '1.3 识别愤怒', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '二、信息提取', '三级标题': '2.1 商品信息提取', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '二、信息提取', '三级标题': '2.2 综合情感推断和信息提取', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标题': '三、主题推断', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}
{'一级标题': '第五章 推断', '二级标

In [8]:
#继续按照字符串内进行分割
CHUNK_SIZE=500
OVERLAP_SIZE=50

text_spliter=RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=OVERLAP_SIZE)
final_docs=text_spliter.split_documents(header_docs)

In [9]:
print(final_docs)

[Document(metadata={'一级标题': '第五章 推断', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}, page_content='在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。  \n让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的情感和主题。这些任务包括了标签提取、实体提取、以及理解文本的情感等等。在传统的机器学习流程中，你需要收集标签化的数据集、训练模型、确定如何在云端部署模型并进行推断。尽管这种方式可能会产生不错的效果，但完成这一全流程需要耗费大量的时间和精力。而且，每一个任务，比如情感分析、实体提取等等，都需要训练和部署单独的模型。  \n然而，就在你准备投入繁重工作的时候，你发现了大型语言模型（LLM）。LLM 的一个明显优点是，对于许多这样的任务，你只需要编写一个 Prompt，就可以开始生成结果，大大减轻了你的工作负担。这个发现像是找到了一把神奇的钥匙，让应用程序开发的速度加快了许多。最令你兴奋的是，你可以仅仅使用一个模型和一个 API 来执行许多不同的任务，无需再纠结如何训练和部署许多不同的模型。  \n让我们开始这一章的学习，一起探索如何利用 LLM 加快我们的工作进程，提高我们的工作效率。'), Document(metadata={'一级标题': '第五章 推断', '二级标题': '一、情感推断', '三级标题': '1.1 情感倾向分析', 'source': './llm-universe/data_base/knowledge_db/prompt_engineering/5. 推断 Inferring.md'}, page_content='让我们以一则电商平台上的台灯评论为例，通过此例，我们将学习如何对评论进行情感二分类（正面/负面）。  \n```python\nlamp_review = """\n我需要一盏漂亮的卧室灯，这款灯具有额外的储物功能，价格也不算太高。\\\n我很快就收到了它。在运输过程中，我们的灯绳断了，但是公司很乐意寄送了一个新的。\\\n几天后就收到了。这款灯很容易组装。我发

In [ ]:
# #构建向量库
# embedding = ZhipuAIEmbeddings()
# persist_directory="./vector_db/chroma"#定义持久化路径
# vectordb=Chroma.from_documents(
#     documents=final_docs,
#     embedding=embedding,
#     persist_directory=persist_directory
# )
# print(vectordb._collection.count())#向量库中存储的数量

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


32


In [17]:
#向量检索
question="什么是情感推断？"
sim_docs=vectordb.similarity_search(question,k=3)
for doc in sim_docs:
    print(doc.page_content)
    print("==============================")

在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。  
让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的情感和主题。这些任务包括了标签提取、实体提取、以及理解文本的情感等等。在传统的机器学习流程中，你需要收集标签化的数据集、训练模型、确定如何在云端部署模型并进行推断。尽管这种方式可能会产生不错的效果，但完成这一全流程需要耗费大量的时间和精力。而且，每一个任务，比如情感分析、实体提取等等，都需要训练和部署单独的模型。  
然而，就在你准备投入繁重工作的时候，你发现了大型语言模型（LLM）。LLM 的一个明显优点是，对于许多这样的任务，你只需要编写一个 Prompt，就可以开始生成结果，大大减轻了你的工作负担。这个发现像是找到了一把神奇的钥匙，让应用程序开发的速度加快了许多。最令你兴奋的是，你可以仅仅使用一个模型和一个 API 来执行许多不同的任务，无需再纠结如何训练和部署许多不同的模型。  
让我们开始这一章的学习，一起探索如何利用 LLM 加快我们的工作进程，提高我们的工作效率。
接下来，我们将继续使用之前的台灯评论，但这次我们会试用一个新的 Prompt 。我们希望模型能够识别出评论作者所表达的情感，并且将这些情感整理为一个不超过五项的列表。  
```python

prompt = f"""
识别以下评论的作者表达的情感。包含不超过五个项目。将答案格式化为以逗号分隔的单词列表。

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
```  
满意,感激,赞赏,信任,满足  
大型语言模型非常擅长从一段文本中提取特定的东西。在上面的例子中，评论所表达的情感有助于了解客户如何看待特定的产品。
大型语言模型的另一个很酷的应用是推断主题。假设我们有一段长文本，我们如何判断这段文本的主旨是什么？它涉及了哪些主题？让我们通过以下一段虚构的报纸报道来具体了解一下。  
```python

story = """
在政府最近进行的一项调查中，要求公共部门的员工对他们所在部门的满意度进行评分。
调查结果显示，NASA 是最受欢迎的部门，满意度为 95％。

一位 NASA 员工 Joh

In [16]:
#使用MMR最大边际相关性检索
mmr_docs = vectordb.max_marginal_relevance_search(question,k=3)
for doc in mmr_docs:
    print(doc.page_content)
    print("==============")

在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。  
让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的情感和主题。这些任务包括了标签提取、实体提取、以及理解文本的情感等等。在传统的机器学习流程中，你需要收集标签化的数据集、训练模型、确定如何在云端部署模型并进行推断。尽管这种方式可能会产生不错的效果，但完成这一全流程需要耗费大量的时间和精力。而且，每一个任务，比如情感分析、实体提取等等，都需要训练和部署单独的模型。  
然而，就在你准备投入繁重工作的时候，你发现了大型语言模型（LLM）。LLM 的一个明显优点是，对于许多这样的任务，你只需要编写一个 Prompt，就可以开始生成结果，大大减轻了你的工作负担。这个发现像是找到了一把神奇的钥匙，让应用程序开发的速度加快了许多。最令你兴奋的是，你可以仅仅使用一个模型和一个 API 来执行许多不同的任务，无需再纠结如何训练和部署许多不同的模型。  
让我们开始这一章的学习，一起探索如何利用 LLM 加快我们的工作进程，提高我们的工作效率。
prompt = f"""
以下用三个反引号分隔的产品评论的情感是什么？

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
```  
情感是积极的。  
如果你想要给出更简洁的答案，以便更容易进行后期处理，可以在上述 Prompt 基础上添加另一个指令：*用一个单词回答：「正面」或「负面」*。这样就只会打印出 “正面” 这个单词，这使得输出更加统一，方便后续处理。  
```python
prompt = f"""
以下用三个反引号分隔的产品评论的情感是什么？

用一个单词回答：「正面」或「负面」。

评论文本: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
```  
正面
```python
from tool import get_completion
